# Task 5 — Source Metadata Ingestion into MongoDB

## Yêu cầu của đề

Spark Structured Streaming đọc metadata từ Kafka, parse JSON bằng schema rõ
ràng, ghi MongoDB và dùng checkpoint để tiếp tục đúng offset sau restart.

## Cách triển khai và lý do

[`task5/metadata_stream.py`](https://github.com/linh285/huggingface-cpg-streaming/blob/fix/lab04-final-integration/task5/metadata_stream.py) dùng
`StructType`, chỉ nhận `FILE_METADATA_UPSERT` schema `1.0.0`, thêm Kafka
partition/offset và ghi từng micro-batch bằng replace/upsert. `_id=file_id` làm
khóa duy nhất; checkpoint ở `/opt/spark-checkpoints/cpg-metadata` được gắn vào
named Docker volume.

## Output thật đã chạy


In [1]:
import json
from pathlib import Path

proof = json.loads(Path("../artifacts/task5/mongodb_verification.json").read_text(encoding="utf-8"))
print(json.dumps(proof, indent=2))


{
  "status": "PASS",
  "stream_container_state": "running",
  "stream_started": true,
  "checkpoint_location": "/opt/spark-checkpoints/cpg-metadata",
  "checkpoint_file_count": 48,
  "mongodb_database": "cpg",
  "mongodb_collection": "source_metadata",
  "mongo": {
    "count_by_id": 1,
    "count_by_file_id": 1,
    "document": {
      "_id": "6c7f76928a88fa4fd8217323002eb245416159a35a66ca6559463976f8d1fc9c",
      "file_id": "6c7f76928a88fa4fd8217323002eb245416159a35a66ca6559463976f8d1fc9c",
      "path": "src/datasets/utils/experimental.py",
      "content_sha256": "6acb052a2ded7705891b4be5a06fba143eb2881e75612fa606bc31fe8ef677b3",
      "kafka_partition": 0,
      "kafka_offset": 11
    }
  }
}


## Bằng chứng Spark và MongoDB

![Spark Structured Streaming query](images/task5/spark-structured-streaming.png)

![MongoDB source_metadata document](images/task5/mongodb-source-metadata.png)

Mongo Express chỉ là profile UI tùy chọn để chụp bằng chứng; pipeline mặc định
không phụ thuộc vào service này.

## Lỗi đã gặp và cách khắc phục

Lần đầu Spark cần tải connector packages nên query khởi động chậm. Nhóm dùng
named Ivy cache và healthcheck thay vì coi container `Up` là stream đã sẵn sàng.
Pull Mongo Express từng lỗi DNS; service được chuyển sang profile `ui` để lỗi UI
không làm full stack thất bại.

## Reflection

Checkpoint bảo vệ offset, còn MongoDB upsert bảo vệ dữ liệu. Chỉ một trong hai
không đủ để chứng minh exactly-once effect ở collection.

## Tái lập

```bash
docker compose -f compose.yml -f task4/docker-compose.yml -f task5/docker-compose.yml up -d
docker compose -f compose.yml -f task4/docker-compose.yml -f task5/docker-compose.yml \
  exec -T mongodb mongosh --quiet cpg --eval 'db.source_metadata.countDocuments()'
docker compose -f compose.yml -f task4/docker-compose.yml -f task5/docker-compose.yml \
  exec -T metadata-stream sh -c 'find /opt/spark-checkpoints/cpg-metadata -type f | wc -l'
```
